In [0]:
import plotly.graph_objects as go

df = spark.sql("""
WITH l AS (
  SELECT
      codigo_loja,
      tipo_loja     as tipo,
      status_loja   as status,
      cluster_loja  as cluster
  FROM ds_dev.cvc_pred.lojas
  WHERE codigo_loja IS NOT NULL
),
base AS (
  SELECT 
    f.codigo_loja,
    upper(f.canal) as canal,
    f.data_previsao as data,
    CASE dayofweek(f.data_previsao)
        WHEN 1 THEN 'domingo'
        WHEN 2 THEN 'segunda'
        WHEN 3 THEN 'terça'
        WHEN 4 THEN 'quarta'
        WHEN 5 THEN 'quinta'
        WHEN 6 THEN 'sexta'
        WHEN 7 THEN 'sábado'
    END as dia_semana,
    -- Agora pegamos os campos numéricos já calculados na tabela de reconciliação
    f.forecast,
    f.lower,
    f.upper
  FROM ds_dev.cvc_pred.previsao_venda_reconciliada_final as f
  LEFT JOIN l ON f.codigo_loja = l.codigo_loja
  WHERE l.status = 'ATIVO'
    AND upper(f.canal) = 'LOJAS'
)
SELECT
  data,
  dia_semana,
  ROUND(SUM(forecast), 2) as forecast,
  ROUND(SUM(lower), 2)    as lower,
  ROUND(SUM(upper), 2)    as upper
FROM base
GROUP BY data, dia_semana
ORDER BY data
""")

pdf = df.toPandas()
pdf["data"] = pdf["data"].astype("datetime64[ns]")
pdf = pdf.sort_values("data")

fig = go.Figure()

# Linha do forecast
fig.add_trace(go.Scatter(
    x=pdf["data"], y=pdf["forecast"],
    mode="lines",
    name="Forecast",
    line=dict(width=2)
))

# Banda de intervalo [lower, upper]
fig.add_trace(go.Scatter(
    x=pdf["data"],
    y=pdf["upper"],
    mode="lines",
    line=dict(width=0),
    showlegend=False,
    name="Upper"
))

fig.add_trace(go.Scatter(
    x=pdf["data"],
    y=pdf["lower"],
    mode="lines",
    fill="tonexty",
    fillcolor="rgba(0, 0, 0, 0.12)",
    line=dict(width=0),
    showlegend=True,
    name="Intervalo"
))
fig.update_layout(
    title="Forecast (soma das lojas) por dia — canal LOJAS",
    xaxis_title="Data",
    yaxis_title="Forecast",
    hovermode="x unified",
    template="plotly_white",
    height=420
)

fig.show()